# 08 Image Level Evaluation

In [ ]:
!rm -rf /content/unet-ensemble

In [1]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

Cloning into 'unet-ensemble'...
remote: Enumerating objects: 464, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 464 (delta 95), reused 71 (delta 35), pack-reused 313 (from 1)
Receiving objects: 100% (464/464), 153.66 KiB | 3.49 MiB/s, done.
Resolving deltas: 100% (276/276), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
sys.path.append('/content/unet-ensemble')

In [4]:
!pip install safetensors huggingface_hub segmentation-models-pytorch scikit-image scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.2 MB/s eta 0:00:00


In [5]:
import os
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from huggingface_hub import login
from torch.utils.data import DataLoader

from src.dataset.dataset    import Dataset as LazyDS
from src.dataset.dataloader import DataLoader as DSLoader
from src.eval.evaluate      import Evaluate
from src.eval.metrics       import Metrics

Metric helpers defined.


In [ ]:
token = '' 
login(token=token)

## 08-1 Configuration

In [7]:
# Dataset 
IMG_SIZE     = 256
DATASET_ROOT = '/content/drive/Shareddrives/U-Net Ensemble/Dataset'
BATCH_SIZE   = 1
NUM_WORKERS  = 2

MASK_FOLDER         = 'GTA - Masks'
PRNU_FOLDER         = 'Feature - PRNU'
ILLUMINATION_FOLDER = 'Feature - Illumination'
FREQUENCY_FOLDER    = 'Feature - Frequency'
CATEGORIES          = ['Synthetic', 'Authentic']
TEMPLATES           = [
    'template-a', 'template-albania', 'template-b',
    'template-c', 'template-chile',   'template-deutschland',
    'template-spain', 'template-usa',
]

# Feature combination 
# Must match the combination used when training the checkpoint.
# Options: ('prnu','illumination','frequency') | ('prnu','frequency')
#          ('prnu','illumination')             | ('frequency','illumination')
FEATURES = ('prnu', 'illumination', 'frequency')

# HuggingFace repos 
HF_USERNAME  = 'Jesayyy'
MODEL_REPO  = f'{HF_USERNAME}/mben-unetpp'
SUBFOLDER    = 'all_features'   # change to match the checkpoint subfolder

# Output CSV path (Google Drive) 
CSV_DIR  = '/content/drive/Shareddrives/U-Net Ensemble/checkpoint/U-Net++ (All)'
CSV_PATH = os.path.join(CSV_DIR, 'per_image_evaluation_results.csv')

# Misc 
THRESHOLD      = 0.5
BOUNDARY_WIDTH = 2       # dilation tolerance for BF Score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'Features: {sorted(FEATURES)}')

Device  : cuda
Features: ['frequency', 'illumination', 'prnu']


## 08-2 U-Net++ / Attention U-Net

### A. Dataset Preparation

In [8]:
ds_loader = DSLoader(
    mask_folder         = MASK_FOLDER,
    prnu_folder         = PRNU_FOLDER,
    illumination_folder = ILLUMINATION_FOLDER,
    frequency_folder    = FREQUENCY_FOLDER,
    categories          = CATEGORIES,
    templates           = TEMPLATES,
    features            = FEATURES,
)

eval_samples = ds_loader.load_images('Evaluation', DATASET_ROOT)
eval_ds = LazyDS(eval_samples, IMG_SIZE, augment=False, features=FEATURES)

# batch_size=1 is required so each row in the CSV corresponds to exactly one image
eval_loader = DataLoader(
    eval_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
)

print(f'Features     : {sorted(FEATURES)}')
print(f'Total images : {len(eval_samples)}')
print(f'Total batches: {len(eval_loader)}')

[DataLoader] Active features: ['frequency', 'illumination', 'prnu']
[Evaluation] Total samples found: 960
Features     : ['frequency', 'illumination', 'prnu']
Total images : 960
Total batches: 960


### B. Load Model

In [9]:
evaluator = Evaluate(device=device, features=FEATURES, threshold=THRESHOLD,
                     boundary_width=BOUNDARY_WIDTH)

model  = evaluator.load_unetpp_from_hub(MODEL_REPO,  SUBFOLDER)\

print('Both models loaded and ready.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

all_features/model.safetensors:   0%|          | 0.00/84.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

[UNet++] Loaded from Jesayyy/mben-unetpp/all_features
Both models loaded and ready.


### C. Inference Loop

In [ ]:
FEATURE_ORDER   = ('prnu', 'illumination', 'frequency')
active_features = [f for f in FEATURE_ORDER if f in frozenset(FEATURES)]

rows = []

model.eval()

with torch.no_grad():
    for img_idx, batch in enumerate(tqdm(eval_loader, desc='Evaluating per image')):

        # Unpack batch 
        # batch = (*feature_tensors_in_canonical_order, fused_t, mask_t)
        *feat_tensors, fused, masks = batch

        feature_dict = {
            feat: feat_tensors[i].to(device)
            for i, feat in enumerate(active_features)
        }
        fused = fused.to(device)
        masks = masks.to(device)

        # Grab the source file path for this sample (for the CSV) 
        sample_path = eval_samples[img_idx].get('mask', f'image_{img_idx:05d}')
        image_name  = os.path.basename(sample_path)

        gt_np = masks.cpu().numpy()[0, 0].astype(np.uint8) 

        logits_unetpp = model(feature_dict, fused)   # (1, 1, H, W)
        prob_unetpp   = torch.sigmoid(logits_unetpp).cpu().numpy()[0, 0]  # (H, W)
        pred_unetpp   = (prob_unetpp >= THRESHOLD).astype(np.uint8)

        m = Metrics()
        metrics = m.compute_all_metrics(prob_unetpp, pred_unetpp, gt_np, BOUNDARY_WIDTH)

        # Collect row 
        row = {
            'image_index' : img_idx,
            'image_name'  : image_name,
            'features'    : '+'.join(sorted(FEATURES)),

            # U-Net++ metrics
            'Feature_Correlation': metrics['Feature_Correlation'],
            'Dice'               : metrics['Dice'],
            'IoU'                : metrics['IoU'],
            'Pixel_Accuracy'     : metrics['Pixel_Accuracy'],
            'BF_Score'           : metrics['BF_Score']
        }
        rows.append(row)

print(f'\nInference complete. {len(rows)} images evaluated.')

Evaluating per image:   0%|          | 0/960 [00:05<?, ?it/s]


AttributeError: 'numpy.ndarray' object has no attribute 'feature_correlation'

## 08-3 Baseline

## 08-4 Build Dataframe & Preview

In [ ]:
results_df = pd.DataFrame(rows)

pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', None)

print(f'Shape: {results_df.shape}')
results_df.head(10)

## 08-5 Aggregate Summary

In [ ]:
metric_cols = ['Feature_Correlation', 'Dice', 'IoU', 'Pixel_Accuracy', 'BF_Score']

summary_rows = []
for model_prefix in ('unetpp', 'attunet'):
    label = 'U-Net++' if model_prefix == 'unetpp' else 'Attention U-Net'
    cols  = [f'{model_prefix}_{m}' for m in metric_cols]
    means = results_df[cols].mean()
    row   = {'Model': label}
    for m, col in zip(metric_cols, cols):
        row[m] = round(means[col], 4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print('\n── Macro-averaged metrics ──')
display(summary_df)

## 08-6 Save CSV to Drive

In [ ]:
os.makedirs(CSV_DIR, exist_ok=True)

if os.path.exists(CSV_PATH):
    # Append without re-writing the header
    results_df.to_csv(CSV_PATH, mode='a', header=False, index=False)
    print(f'Results appended to existing file: {CSV_PATH}')
else:
    results_df.to_csv(CSV_PATH, mode='w', header=True, index=False)
    print(f'Results saved to new file: {CSV_PATH}')

print(f'Total rows written: {len(results_df)}')
print(f'Total columns     : {len(results_df.columns)}')